# Analisando dados da NBA

Peguei dados do site https://www.kaggle.com/datasets/flynn28/v2-nba-player-database/data que diz que extraiu dados do site https://www.basketball-reference.com/

Estou querendo analisar as alturas e sei que do Wembanyama é 224cm, porém no Kaggle ele aparece como 221 cm enquanto que no reference aparece como 224. Depois descobri que ele foi listado inicialmente como 7'3" (87 polegadas/221 cm). Depois, medições oficiais da NBA o colocaram como 7'4" (88 polegadas/224 cm).
Vamos ter que pegar dados da nba direto.

In [12]:
# Passo a passo
# Passo 1: Importar base de dados
# Passo 2: Visualizar base de dados
# Passo 3: Converter alturas em centímetros
# Passo 4: Fazer gráfico das alturas

# Passo a passo do projeto
# Passo 1: Importar a base de dados
import pandas as pd
from nba_api.stats.endpoints import playerindex

# temporada atual (ex.: 2025-26); pode fixar: season="2025-26"
indice = playerindex.PlayerIndex(
    season="2025-26",      # ajuste se precisar
    active_nullable="1",   # só jogadores ativos
    league_id="00",        # NBA
    timeout=60,
)

df = indice.player_index.get_data_frame()

df["nome"] = df["PLAYER_FIRST_NAME"] + " " + df["PLAYER_LAST_NAME"]

jogadores = df[["nome", "HEIGHT"]].rename(columns={"HEIGHT": "height"})
jogadores.to_csv("nba_ativos.csv", index=False)

dados = pd.read_csv(r"D:\cursos\python\estatisticas\nba\nba_ativos.csv")

# Passo 2: Visualizar a base de dados
display(dados)
# colunas inúteis - informações que não te ajudam, te atrapalham
display(dados.info())


,nome,height
0,Precious Achiuwa,6-8
1,Steven Adams,6-11
2,Bam Adebayo,6-9
3,Ochai Agbaji,6-5
4,Santi Aldama,7-0
...,...,...
525,Trae Young,6-2
526,Chris Youngblood,6-4
527,Rocco Zikarsky,7-3
528,Ivica Zubac,7-0


<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   nome    530 non-null    str  
 1   height  530 non-null    str  
dtypes: str(2)
memory usage: 8.4 KB


None

In [13]:
# Passo 3: Análise das alturas
# Converter alturas em centímetros
def altura_para_cm(height):
    if pd.isna(height) or height == "":
        return None
    pes, pol = map(int, str(height).split("-"))
    return round((pes * 12 + pol) * 2.54, 0)

dados["altura"] = dados["height"].apply(altura_para_cm)
display(dados["altura"].value_counts().sort_index())


altura
170.0     1
180.0     3
183.0    12
185.0    17
188.0    28
190.0    28
193.0    52
196.0    66
198.0    58
201.0    60
203.0    57
206.0    45
208.0    35
211.0    29
213.0    20
216.0    13
218.0     3
221.0     2
224.0     1
Name: count, dtype: int64

In [14]:
# Passo 4: Fazer gráfico das alturas
import plotly.express as px

# criar o grafico
grafico = px.histogram(dados, x="altura")
# exibir o grafico
grafico.show()

In [16]:
# tirar informações específicas dos dados
display(dados[dados["altura"]<183][["nome", "altura", "height"]].sort_values(by="altura"))
display(dados[dados["altura"] > 216][["nome", "altura", "height"]].sort_values(by="altura"))


,nome,altura,height
251,Yuki Kawamura,170.0,5-7
309,Jordan McLaughlin,180.0,5-11
341,Ryan Nembhard,180.0,5-11
448,Isaiah Stevens,180.0,5-11


,nome,altura,height
84,Donovan Clingan,218.0,7-2
255,Walker Kessler,218.0,7-2
374,Kristaps Porziņģis,218.0,7-2
123,Zach Edey,221.0,7-3
527,Rocco Zikarsky,221.0,7-3
499,Victor Wembanyama,224.0,7-4


Agora vamos pegar os dados de altura do brasileiro pela PNS de 2019 do IBGE
https://www.ibge.gov.br/estatisticas/sociais/trabalho/9160-pesquisa-nacional-de-saude.html?utm_source=openai&t=microdados

O arquivo tem mais de 400MB, então extraí somente as colunas que preciso.

In [ ]:
import pandas as pd
# Configurações de posições (subtraímos 1 pois o Python começa em 0)
# SAS @108  tamanho 1 -> Python (107, 108)
# SAS @117  tamanho 3 -> Python (116, 119)
# SAS @1362 tamanho 5 -> Python (1361, 1366)

colunas_posicoes = [
    (107, 108),   # C006 - Sexo
    (116, 119),   # C008 - Idade
    (1361, 1366)  # W00203 - Altura
]

nomes_colunas = ['SEXO', 'IDADE', 'ALTURA_CM']

print("Lendo arquivo PNS... aguarde.")

# Lê o arquivo TXT usando as posições fixas
df = pd.read_fwf(
    r'C:\Users\User\Desktop\PNS_2019.txt',
    colspecs=colunas_posicoes,
    names=nomes_colunas,
    dtype=str
)

# 1. Converter IDADE para número
df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce')

# 2. Tratar a ALTURA
df['ALTURA_CM'] = pd.to_numeric(df['ALTURA_CM'], errors='coerce')

# Filtrar o código 999 (Erro) e valores vazios antes de calcular
df = df[df['ALTURA_CM'] < 999]

# 3. Criar arquivo CSV final apenas com o que interessa
# Opcional: filtrar apenas homens (SEXO == '1') para comparar com a NBA
df_final = df[['SEXO', 'IDADE', 'ALTURA_CM']].dropna()

df_final.to_csv('alturas_pns_limpo.csv', index=False)

print(f"Sucesso! Arquivo 'alturas_pns_limpo.csv' gerado com {len(df_final)} registros.")
print(df_final.head())

#sexo: 1 = homem, 2 = mulher

Lendo arquivo PNS... aguarde.
Sucesso! Arquivo 'alturas_pns_limpo.csv' gerado com 6730 registros.
     SEXO  IDADE  ALTURA_CM
1142    1     47      169.0
1147    2     71      138.0
1152    1     44      170.0
1156    2     56      155.0
1158    2     58      162.0


In [21]:
# Passo 1: Importar a base de dados
import pandas as pd

dadosIBGE = pd.read_csv(r"D:\cursos\python\estatisticas\nba\alturas_pns_limpo.csv")

# Passo 2: Visualizar a base de dados
display(dadosIBGE)
# colunas inúteis - informações que não te ajudam, te atrapalham
display(dadosIBGE.info())

,SEXO,IDADE,ALTURA_CM
0,1,47,169.0
1,2,71,138.0
2,1,44,170.0
3,2,56,155.0
4,2,58,162.0
...,...,...,...
6725,2,48,143.5
6726,2,35,168.7
6727,2,44,155.4
6728,2,30,160.4


<class 'pandas.DataFrame'>
RangeIndex: 6730 entries, 0 to 6729
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   SEXO       6730 non-null   int64  
 1   IDADE      6730 non-null   int64  
 2   ALTURA_CM  6730 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 157.9 KB


None

In [36]:
#pegar só homens
homens_ibge = dadosIBGE[dadosIBGE["SEXO"] == 1]
display(homens_ibge["ALTURA_CM"].value_counts().sort_index())
display(homens_ibge["IDADE"].value_counts().sort_index())

homens_jovens = homens_ibge[homens_ibge["IDADE"].between(20, 35)]
display(homens_jovens["ALTURA_CM"].value_counts().sort_index())

ALTURA_CM
141.0    1
143.5    1
144.0    1
144.1    1
144.7    1
        ..
193.0    2
193.4    1
194.8    1
195.0    1
195.1    1
Name: count, Length: 367, dtype: int64

IDADE
15     18
16     42
17     35
18     51
19     31
       ..
93      1
95      2
96      1
97      1
104     1
Name: count, Length: 83, dtype: int64

ALTURA_CM
143.5    1
153.0    3
153.3    1
153.5    1
153.6    1
        ..
191.6    1
193.0    1
193.4    1
194.8    1
195.1    1
Name: count, Length: 221, dtype: int64

In [28]:
# criar o grafico
grafico2 = px.histogram(homens_jovens, x="ALTURA_CM")
# exibir o grafico
grafico2.show()

In [40]:
import plotly.graph_objects as go

fig_comparacao = go.Figure()

fig_comparacao.add_trace(
    go.Histogram(x=dados["altura"], name="NBA ativos (2026)", opacity=0.6)
)
fig_comparacao.add_trace(
    go.Histogram(x=homens_jovens["ALTURA_CM"], name="Brasileiros jovens (2019)", opacity=0.6)
)

fig_comparacao.update_layout(
    width=700,          # menos largo (ajuste: 600, 650...)
    height=500,
    barmode="overlay",
    title="Alturas de jogadores ativos na NBA e brasileiros de 20 a 35 anos",
    xaxis_title="Altura (cm)",
    yaxis_title="Quantidade",
    legend=dict(
        orientation="h",      # legenda na horizontal
        yanchor="top",
        y=-0.15,              # abaixo do gráfico (mais negativo = mais embaixo)
        xanchor="center",
        x=0.5,                # centralizada
        title_text="",
    ),
    margin=dict(b=100),       # espaço embaixo para a legenda não cortar
)

fig_comparacao.show()

In [35]:
#informações importantes
display(homens_jovens["ALTURA_CM"].describe())
display(dados["altura"].describe())

count    859.000000
mean     173.413271
std        7.290284
min      143.500000
25%      168.750000
50%      173.000000
75%      178.500000
max      195.100000
Name: ALTURA_CM, dtype: float64

count    530.000000
mean     199.583019
std        8.368729
min      170.000000
25%      193.000000
50%      199.500000
75%      206.000000
max      224.000000
Name: altura, dtype: float64

In [44]:
# algumas comparações
# 1. Calcular o total de jogadores
total_jogadores = len(dados)

# 2. Contar quantos estão acima de 173 cm
acima_173 = len(dados[dados["altura"] > 173])

# 3. Calcular a porcentagem
porcentagem = (acima_173 / total_jogadores) * 100

print(f"Total de jogadores: {total_jogadores}")
print(f"Jogadores acima de 173 cm: {acima_173}")
print(f"Porcentagem: {porcentagem:.2f}%")

percentual = (dados["altura"] > 205).mean() * 100
print(f"{percentual:.2f}% dos jogadores da NBA têm mais de 205 cm.")

percentual = (dados["altura"] > 195).mean() * 100
print(f"{percentual:.2f}% dos jogadores da NBA têm mais de 195 cm.")

Total de jogadores: 530
Jogadores acima de 173 cm: 529
Porcentagem: 99.81%
27.92% dos jogadores da NBA têm mais de 205 cm.
73.40% dos jogadores da NBA têm mais de 195 cm.
